# SimuSIGW Tutorial

This notebook is a compact usage guide for the current `SimuSIGW` package.

It uses ASCII text only in markdown cells to avoid notebook font or encoding issues. The examples use small grids by default so that the notebook can be run interactively.

## 1. Import the package

If you installed the package with `pip install -e .`, the direct import should work. The path insertion below is only a convenience when running this notebook from the project root.

In [1]:
from pathlib import Path
import sys
import time
import tempfile

import numpy as np

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import SimuSIGW as sg

print("SimuSIGW version:", sg.__version__)
# print("Project root:", PROJECT_ROOT)

SimuSIGW version: 0.1.0


## 2. Backend overview

The package exposes four evolution entry points:

- `run_evolution_fd`: CPU finite-difference backend, with `fd_order=2` or `fd_order=4`.
- `run_evolution`: CPU FFT pseudospectral backend.
- `run_evolution_torch_fft`: PyTorch FFT backend, recommended for CUDA runs.
- `run_evolution_torch`: PyTorch dense Fourier-matrix backend, kept for compatibility with the original notebook.

Recommended quick choices:

- Fast exploratory CPU runs: `run_evolution_fd`, `fd_order=2`, `dtype=np.float32`.
- More accurate CPU spectral reference: `run_evolution`.
- GPU spectral runs: `run_evolution_torch_fft`, `dtype=np.float32` or `np.float64`.

## 3. Create initial fields

`gaussian_random_fields` returns the Gaussian field and the transformed initial scalar field. The default power spectrum and logarithmic non-Gaussian transform match the package defaults.

In [2]:
dtype = np.float64
n = 32
seed = 1234

_, phi_initial = sg.gaussian_random_fields(
    n,
    power_spectrum=sg.gaussian_bump_power_spectrum,
    non_gaussianity=0.1,
    seed=seed,
    workers=-1,
    dtype=dtype,
)

print(phi_initial.shape, phi_initial.dtype)
print("mean/std:", float(phi_initial.mean()), float(phi_initial.std()))

(32, 32, 32) float64
mean/std: 0.013876481950199573 0.07086394945680106


## 4. Build an `EvolutionConfig`

The physical box convention is inherited from the original notebook: box length is `space_length * pi`, and the default time step is `space_length / n / 5`.

For notebook testing, keep `max_steps` small. For production runs, increase `n`, `max_steps`, and `output_every`.

In [3]:
run_dir = Path("log/tutorial_fd")

config = sg.EvolutionConfig(
    n=n,
    space_length=2.0,
    max_steps=5,
    output_every=0,
    output_path=run_dir,
    workers=-1,
    fft_workers=1,
    fd_order=2,
    dtype=dtype,
    save_initial=False,
)

print(config)
print("dt:", config.dt)
print("initial_time:", config.initial_time)

EvolutionConfig(n=32, space_length=2.0, max_steps=5, output_every=0, output_path=WindowsPath('log/tutorial_fd'), workers=-1, fft_workers=1, fd_order=2, dtype=<class 'numpy.float64'>, save_initial=False)
dt: 0.0125
initial_time: 0.0625


## 5. Run the finite-difference CPU backend

This is usually the fastest CPU option. Use `fd_order=2` for speed and `fd_order=4` for a higher-order stencil. Compare against the FFT backend before trusting production spectra.

In [4]:
sg.warmup_fd_backend(n=8, workers=config.workers, order=config.fd_order, dtype=config.dtype)

start = time.perf_counter()
state_fd = sg.run_evolution_fd(phi_initial, config, warmup=False)
elapsed = time.perf_counter() - start

print("FD final t:", state_fd.t)
print("FD elapsed:", elapsed, "s")

FD final t: 0.12499999999999999
FD elapsed: 0.009825800079852343 s


## 6. Run the CPU FFT backend

This backend uses pseudospectral derivatives. The current version uses rFFT for safe Laplacian operations and spectral RK4 for the scalar field.

In [5]:
config_fft = sg.EvolutionConfig(
    n=n,
    space_length=2.0,
    max_steps=5,
    output_every=0,
    output_path="log/tutorial_fft",
    workers=-1,
    fft_workers=1,
    dtype=dtype,
    save_initial=False,
)

start = time.perf_counter()
state_fft = sg.run_evolution(phi_initial, config_fft)
elapsed = time.perf_counter() - start

print("FFT final t:", state_fft.t)
print("FFT elapsed:", elapsed, "s")

FFT final t: 0.12499999999999999
FFT elapsed: 0.04797880002297461 s


## 7. Compare two states

The finite-difference and FFT backends are different numerical methods, so they are not expected to match exactly. This helper gives a quick scale for differences.

In [6]:
def max_state_difference(a, b):
    fields = (
        "phi", "pi",
        "h11", "h12", "h13", "h22", "h23", "h33",
        "v11", "v12", "v13", "v22", "v23", "v33",
    )
    return max(np.max(np.abs(getattr(a, name) - getattr(b, name))) for name in fields)

print("FD vs FFT max absolute difference:", max_state_difference(state_fd, state_fft))

FD vs FFT max absolute difference: 1.9878126647499132


## 8. Optional: run the PyTorch FFT backend

Use this cell in an environment with PyTorch. On this project, the `simu` conda environment has CUDA support.

Precision can be selected directly with `dtype=np.float32` or `dtype=np.float64`. If `dtype` is omitted, the backend uses `config.dtype`.

In [7]:
try:
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype = np.float32

    config_torch = sg.EvolutionConfig(
        n=n,
        max_steps=5,
        output_every=0,
        output_path="log/tutorial_torch_fft",
        dtype=torch_dtype,
        save_initial=False,
    )

    _, phi_torch = sg.gaussian_random_fields(
        config_torch.n,
        power_spectrum=sg.gaussian_bump_power_spectrum,
        non_gaussianity=0.1,
        seed=seed,
        workers=1,
        dtype=torch_dtype,
    )

    if device == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    state_torch = sg.run_evolution_torch_fft(
        phi_torch,
        config_torch,
        device=device,
        dtype=torch_dtype,
        clear_cache=False,
    )
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    print("device:", device)
    print("torch FFT final t:", state_torch.t)
    print("torch FFT elapsed:", elapsed, "s")
    print("output dtype:", state_torch.phi.dtype)
except ImportError as exc:
    print("PyTorch is not installed in this kernel:", exc)
except RuntimeError as exc:
    print("PyTorch backend could not run:", exc)

device: cuda
torch FFT final t: 0.12499999999999999
torch FFT elapsed: 0.11664470005780458 s
output dtype: float32


## 9. Custom model functions

You can provide any vectorized function `power_spectrum(k)` and an optional `non_gaussian_transform(zeta_g)`.

In [8]:
def my_power_spectrum(k):
    kstar = 20.0
    width = 0.1
    amplitude = 1.0e-2
    return (
        amplitude
        * (1.0 / kstar) ** 3
        / (np.sqrt(2.0 * np.pi) * width)
        * np.exp(-((k / kstar - 1.0) ** 2) / (2.0 * width**2))
        * (2.0 * np.pi**2)
    )


def my_non_gaussian_transform(zeta_g):
    return zeta_g + 0.5 * (zeta_g**2 - np.mean(zeta_g**2))

_, phi_custom = sg.gaussian_random_fields(
    n,
    power_spectrum=my_power_spectrum,
    non_gaussian_transform=my_non_gaussian_transform,
    seed=seed,
    workers=-1,
    dtype=dtype,
)

print(phi_custom.shape, phi_custom.dtype)
print("mean/std:", float(phi_custom.mean()), float(phi_custom.std()))

(32, 32, 32) float64
mean/std: -4.7814577885702154e-15 0.041210467685504626


## 10. Save snapshots and compute a GW spectrum

Set `output_every` to a positive value to save snapshot arrays. Then call `load_gw_spectra` on the saved time step.

This cell writes small tutorial files under `log/tutorial_output` and `GW_data/tutorial`.

In [9]:
output_path = Path("log/tutorial_output")

config_output = sg.EvolutionConfig(
    n=32,
    max_steps=2,
    output_every=2,
    output_path=output_path,
    workers=-1,
    fft_workers=1,
    dtype=np.float32,
    save_initial=False,
)

_, phi_output = sg.gaussian_random_fields(
    config_output.n,
    seed=seed,
    workers=1,
    dtype=config_output.dtype,
)

state_output = sg.run_evolution(phi_output, config_output)
spectra = sg.load_gw_spectra(
    [config_output.max_steps],
    raw_path=config_output.output_path,
    output_dir="GW_data/tutorial",
    name="tutorial",
    dt=config_output.dt,
    workers=config_output.workers,
)

spectrum = spectra[config_output.max_steps]
print("final t:", state_output.t)
print("spectrum shape:", spectrum.shape)
print("first values:", spectrum[:8])

final t: 0.0875
spectrum shape: (29,)
first values: [0.00000000e+00 2.55914945e-09 1.61179671e-08 3.92295289e-08
 8.99158478e-08 1.44666590e-07 2.51718268e-07 3.30569960e-07]


## 11. Benchmark scripts

For command-line benchmarks from the project root:

```bash
python benchmark_leapfrog_backends.py
python benchmark_leapfrog_workers.py
python benchmark_torch_backends.py --dtype float32
python benchmark_torch_backends.py --dtype float64 --skip-dense
```

From a CUDA-enabled conda environment, for example:

```bash
conda run -n simu python benchmark_torch_backends.py --dtype float32
```

In [10]:
# Optional: run a lightweight benchmark from inside the notebook.
# This uses tiny settings and is only meant as a smoke test.

def time_backend(label, fn, phi, cfg, **kwargs):
    start = time.perf_counter()
    result = fn(phi, cfg, **kwargs)
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed / cfg.max_steps:.6f} s/step")
    return result

small_cfg = sg.EvolutionConfig(
    n=32,
    max_steps=3,
    output_every=0,
    workers=-1,
    fft_workers=1,
    fd_order=2,
    dtype=np.float32,
    save_initial=False,
)
_, small_phi = sg.gaussian_random_fields(32, seed=seed, workers=1, dtype=np.float32)

sg.warmup_fd_backend(n=8, workers=-1, order=2, dtype=np.float32)
time_backend("FD2", sg.run_evolution_fd, small_phi, small_cfg, warmup=False)
time_backend("CPU FFT", sg.run_evolution, small_phi, small_cfg)

FD2: 0.000897 s/step
CPU FFT: 0.019557 s/step


EvolutionState(t=0.09999999999999999, phi=array([[[-4.90636755e-02,  1.11909263e-01,  3.81871613e-02, ...,
         -3.31764126e-02, -2.45019340e-02, -5.11453991e-02],
        [-3.42094644e-02, -2.40560382e-02, -2.94326655e-02, ...,
          2.69922692e-02,  6.54900273e-02,  1.25245671e-02],
        [ 1.02526422e-01,  9.94069800e-02, -5.68188502e-03, ...,
         -3.90628679e-02,  7.65283371e-02,  4.83237283e-02],
        ...,
        [ 2.29969365e-02, -1.18891089e-02,  2.80879857e-02, ...,
         -1.46079777e-02, -8.42564087e-03,  4.72288796e-02],
        [-6.80059537e-03,  1.43879469e-01,  1.15518519e-01, ...,
          1.28785717e-01, -1.63569135e-02, -3.67794933e-02],
        [-1.16925756e-02, -1.35173109e-02, -1.97534383e-02, ...,
         -1.25189160e-02, -3.92294235e-02, -5.61158217e-02]],

       [[-6.68450611e-02,  5.02996377e-02,  1.10405307e-02, ...,
         -4.00856698e-03, -7.83808493e-03, -5.08353681e-02],
        [-6.63690350e-02, -8.79716378e-03,  1.36750767e-01, .

## 12. Practical notes

- Use `workers=-1` and `fft_workers=1` for the current CPU FFT backend unless a local benchmark shows otherwise.
- Use `dtype=np.float32` for quick exploration; use `np.float64` for reference runs.
- FD is fastest on CPU but is not spectrally accurate. Compare convergence against CPU FFT or torch FFT.
- `run_evolution_torch_fft` is the recommended PyTorch backend. The dense backend is kept mainly for compatibility with the original method.
- Keep notebook test runs small. Move production runs into scripts once settings are fixed.